# Step 30 — Candidate associations for individual-file picks

Run after step 22. Only retained `primary` picks from the merged table are eligible; suppressed picks are not resurrected. This notebook contains the workflow directly and uses the shared configuration helper.

## Association rule and motivation

For each legacy source event ID, find the earliest and latest pick, then search **all waveform-index windows** for one that contains that entire interval. Exactly one enclosing window yields a candidate association; zero or multiple enclosing windows produce a QC row. The filename/event-ID equality is not used as the matching criterion. There is no tolerance expansion in this step.

This is a timing-based candidate match, not verification of a seismic arrival. It does not check whether a pick's full SEED ID exists in the waveform, whether that channel covers the pick, or whether an arrival is visible. File-wide window bounds can span shorter individual channel records. Clock errors and short triggered windows can prevent valid matches. No automatic clock correction is attempted.

Outputs are the source-group/waveform association CSV, a per-pick map with offsets from waveform start, and unresolved-group QC. The offset is calculated from the existing pick timestamp; it does not independently improve or repair that timestamp. Input files remain unchanged; outputs are replaced under the configured external root. Proceed to step 32 while retaining unresolved cases.

## 1. Configuration

Inspect source paths, output paths and tolerances. This cell resolves configuration without running association or plotting.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    merged_picks=P['MERGED_PICK_INDEX'],
    waveform_index=P['WAVEFORM_INDEX'],
    out_event_csv=P['INDIV_WAVEFORM_EVENT_CSV'],
    out_pick_map_csv=P['INDIV_PICK_WAVEFORM_MAP'],
    out_qc_csv=P['INDIV_WAVEFORM_QC'],
)
display(SETTINGS)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/utc


{'merged_picks': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/22_merged_pick_index.csv',
 'waveform_index': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/10_waveform_index.csv',
 'out_event_csv': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/30_individual_waveform_event_index.csv',
 'out_pick_map_csv': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/30_individual_pick_waveform_map.csv',
 'out_qc_csv': '/Users/thompsong/work/Pinatubo1991Processing/utc/pipeline_tmp/qc/30_individual_waveform_qc.csv'}

## 2. Implementation

These definitions make the workflow inspectable here. Defining them does not write files; the run cell below executes them.

In [2]:
from pathlib import Path

import pandas as pd

def run(
    *,
    merged_picks,
    waveform_index,
    out_event_csv,
    out_pick_map_csv,
    out_qc_csv
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        merged_picks=merged_picks,
        waveform_index=waveform_index,
        out_event_csv=out_event_csv,
        out_pick_map_csv=out_pick_map_csv,
        out_qc_csv=out_qc_csv,
    )

    print("=== STEP 30: Associating individual picks with waveform files ===")

    # ------------------------------------------------------------------
    # Load inputs
    # ------------------------------------------------------------------

    df_picks = pd.read_csv(args.merged_picks, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_wav = pd.read_csv(args.waveform_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})

    # ------------------------------------------------------------------
    # Validate inputs
    # ------------------------------------------------------------------

    required_pick_cols = {
        "event_id",
        "pick_id",
        "pick_time",
        "pick_priority",
    }
    missing = required_pick_cols - set(df_picks.columns)
    if missing:
        raise SystemExit(f"Missing required columns in merged picks CSV: {missing}")

    required_wav_cols = {"event_id", "starttime", "endtime"}
    missing = required_wav_cols - set(df_wav.columns)
    if missing:
        raise SystemExit(f"Missing required columns in waveform index CSV: {missing}")

    # ------------------------------------------------------------------
    # Filter to authoritative individual picks ONLY
    # ------------------------------------------------------------------

    df_picks = df_picks[df_picks["pick_priority"] == "primary"].copy()

    if df_picks.empty:
        raise SystemExit("No authoritative individual picks found in STEP 22")

    # ------------------------------------------------------------------
    # Parse times
    # ------------------------------------------------------------------

    df_picks["pick_time"] = pd.to_datetime(
        df_picks["pick_time"], format="mixed", utc=True, errors="coerce"
    )
    df_wav["starttime"] = pd.to_datetime(
        df_wav["starttime"], format="mixed", utc=True, errors="coerce"
    )
    df_wav["endtime"] = pd.to_datetime(
        df_wav["endtime"], format="mixed", utc=True, errors="coerce"
    )

    if df_picks["pick_time"].isna().any():
        raise SystemExit("Unparseable pick_time values in merged picks")

    # ------------------------------------------------------------------
    # Group by authoritative individual event_id (from STEP 22)
    # ------------------------------------------------------------------

    event_rows = []
    pick_map_rows = []
    qc_rows = []

    grouped = df_picks.groupby("event_id", sort=True)

    for event_id, g in grouped:
        tmin = g["pick_time"].min()
        tmax = g["pick_time"].max()

        matches = df_wav[
            (df_wav["starttime"] <= tmin) &
            (df_wav["endtime"] >= tmax)
        ]

        if len(matches) == 1:
            w = matches.iloc[0]
            waveform_event_id = w["event_id"]

            event_rows.append({
                "source_event_id": event_id,
                "waveform_event_id": waveform_event_id,
                "wav_file": w.get("wav_file"),
                "starttime": w["starttime"].isoformat(),
                "endtime": w["endtime"].isoformat(),
                "n_picks": int(len(g)),
                "association_method": "enclosing_waveform",
            })

            for _, p in g.iterrows():
                pick_map_rows.append({
                    "pick_id": p["pick_id"],

                    # Canonical interface for Step 32
                    "waveform_id": waveform_event_id,

                    # Preserve original semantics / provenance
                    "waveform_event_id": waveform_event_id,

                    "time_offset_from_start":
                        (p["pick_time"] - w["starttime"]).total_seconds(),

                    "raw_line": p.get("raw_line"),
                    "raw_lineno": p.get("raw_lineno"),
                    "pha_file": p.get("pha_file"),
                })

        elif len(matches) == 0:
            qc_rows.append({
                "source_event_id": event_id,
                "issue": "no_enclosing_waveform",
            })
        else:
            qc_rows.append({
                "source_event_id": event_id,
                "issue": "multiple_enclosing_waveforms",
                "n_matches": int(len(matches)),
            })

    # ------------------------------------------------------------------
    # Write outputs
    # ------------------------------------------------------------------

    Path(args.out_event_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out_pick_map_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out_qc_csv).parent.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(event_rows).to_csv(args.out_event_csv, index=False)
    pd.DataFrame(pick_map_rows).to_csv(args.out_pick_map_csv, index=False)
    pd.DataFrame(qc_rows).to_csv(args.out_qc_csv, index=False)

    print("\nSTEP 30 COMPLETE")
    print("----------------")
    print(f"Authoritative individual events processed: {len(grouped)}")
    print(f"Associated waveform events:              {len(event_rows)}")
    print(f"QC issues:                              {len(qc_rows)}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Run

This cell reads the configured inputs and replaces configured outputs. Review the summary before continuing.

In [3]:
result = run(**SETTINGS)

=== STEP 30: Associating individual picks with waveform files ===



STEP 30 COMPLETE
----------------
Authoritative individual events processed: 119
Associated waveform events:              119
QC issues:                              0


/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_67948/3995913156.py:29: DtypeWarning: Columns (0: pha_path) have mixed types. Specify dtype option on import or set low_memory=False.
  df_picks = pd.read_csv(args.merged_picks, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})


In [4]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


df_picks


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,pha_path,station,channel,seed_id,...,raw_pick_time,onset,first_motion,weight,clock_regime,clock_offset_s,clock_uncertainty_s,clock_confidence,clock_status,pick_priority
0,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_1,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,...,1991-06-03T07:39:38.490000Z,NaN,U,0.0,H_UTC_PDT,-25200.0,5.0,high,resolved,primary
1,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_2,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,...,1991-06-03T07:39:39.990000Z,NaN,NaN,2.0,H_UTC_PDT,-25200.0,5.0,high,resolved,primary
2,BURZ P 4 910603073938.05,7,91060300,individual_pha_filename,indpha_91060300_3,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUR,EHZ,XB.BUR..EHZ,...,1991-06-03T07:39:38.050000Z,NaN,NaN,4.0,H_UTC_PDT,-25200.0,5.0,high,resolved,primary
3,CABZ P 5 910603073953.23,9,91060300,individual_pha_filename,indpha_91060300_4,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,CAB,EHZ,XB.CAB..EHZ,...,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,H_UTC_PDT,-25200.0,5.0,high,resolved,primary
4,GRNZ P 5 910603073953.23,2,91060300,individual_pha_filename,indpha_91060300_5,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,GRN,EHZ,XB.GRN..EHZ,...,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,H_UTC_PDT,-25200.0,5.0,high,resolved,primary


df_wav


,event_id,dmx_file,wav_file,starttime,endtime,stations,channels,ntraces,status,raw_sampling_rate,sampling_rate,dmx_localtime,dmx_timecorrect,dropped_traces,dropped_trace_ids,raw_starttime,clock_offset_s,clock_regime,partial_recovery,recovery_details
0,9105010W,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-04-30 12:10:09.950000+00:00,1991-04-30 12:10:50.572400+00:00,"PIE,UBO",EHZ,2,existing,100.80645,100.806452,0,0,3,unk.IRIG.._:irig;unk.UBOZ.._:flat;unk.PIEZ.._:...,1991-05-01T03:10:09.950000,-54000.0,H_PHT,False,{}
1,9105010X,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-04-30 12:39:42.521000+00:00,1991-04-30 12:40:58.696680+00:00,"PIE,UBO",EHZ,2,existing,100.80645,100.806452,0,0,3,unk.IRIG.._:irig;unk.UBOZ.._:flat;unk.PIEZ.._:...,1991-05-01T03:39:42.521000,-54000.0,H_PHT,False,{}
2,91050118,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-04-30 19:45:26.647000+00:00,1991-04-30 19:46:12.348440+00:00,"CAB,PIE,UBO",EHZ,3,existing,100.80645,100.806452,0,0,1,unk.IRIG.._:irig,1991-05-01T10:45:26.647000,-54000.0,H_PHT,False,{}
3,91050119,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-04-30 19:49:40.597000+00:00,1991-04-30 19:50:26.298440+00:00,"CAB,PIE,UBO",EHZ,3,existing,100.80645,100.806452,0,0,1,unk.IRIG.._:irig,1991-05-01T10:49:40.597000,-54000.0,H_PHT,False,{}
4,9105011A,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-04-30 20:20:14.116000+00:00,1991-04-30 20:20:59.817440+00:00,"CAB,PIE,UBO",EHZ,3,existing,100.80645,100.806452,0,0,1,unk.IRIG.._:irig,1991-05-01T11:20:14.116000,-54000.0,H_PHT,False,{}


event_rows


[{'source_event_id': '91060300',
  'waveform_event_id': '91060300',
  'wav_file': '/Users/thompsong/work/Pinatubo1991Processing/utc/FAIR/SEISAN/WAV/PNTBO/1991/06/1991-06-03-0039-28M.PNTBO_008',
  'starttime': '1991-06-03T00:39:28.001000+00:00',
  'endtime': '1991-06-03T00:40:18.781480+00:00',
  'n_picks': 11,
  'association_method': 'enclosing_waveform'},
 {'source_event_id': '91060301',
  'waveform_event_id': '91060301',
  'wav_file': '/Users/thompsong/work/Pinatubo1991Processing/utc/FAIR/SEISAN/WAV/PNTBO/1991/06/1991-06-03-0105-32M.PNTBO_008',
  'starttime': '1991-06-03T01:05:32.333000+00:00',
  'endtime': '1991-06-03T01:06:48.508680+00:00',
  'n_picks': 11,
  'association_method': 'enclosing_waveform'},
 {'source_event_id': '91060302',
  'waveform_event_id': '91060302',
  'wav_file': '/Users/thompsong/work/Pinatubo1991Processing/utc/FAIR/SEISAN/WAV/PNTBO/1991/06/1991-06-03-0109-36M.PNTBO_008',
  'starttime': '1991-06-03T01:09:36.125000+00:00',
  'endtime': '1991-06-03T01:10:21.82644

... (2 outputs left out)

qc_rows


[]

matches


,event_id,dmx_file,wav_file,starttime,endtime,stations,channels,ntraces,status,raw_sampling_rate,sampling_rate,dmx_localtime,dmx_timecorrect,dropped_traces,dropped_trace_ids,raw_starttime,clock_offset_s,clock_regime,partial_recovery,recovery_details
2739,9106110O,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/u...,1991-06-11 17:11:06.559000+00:00,1991-06-11 17:12:22.734680+00:00,"BUG,CAB,GRN,PIE",EHZ,4,existing,100.80645,100.806452,-25200,0,5,unk.UBOZ.._:flat;unk.PPOE.._:flat;unk.PPOZ.._:...,1991-06-12T00:11:06.559000,-25200.0,H_UTC_PDT,False,{}
